$\textbf{1: All imports}$

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import pandas as pd

$\textbf{2: Loading model}$

In [3]:
model_name = "./Models/qwen2.5-transformers-0.5b-instruct-gptq-int4-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModel.from_pretrained(model_name)

$\textbf{3: Creating dataset}$

In [4]:
class RedditGuidelineDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = (
            f"Subreddit: {row['subreddit']}\n"
            f"Rule: {row['rule']}\n"
            f"Positive examples: {row['positive_example_1']} || {row['positive_example_2']}\n"
            f"Negative examples: {row['negative_example_1']} || {row['negative_example_2']}\n"
            f"Comment: {row['body']}"
        )
        label = int(row["rule_violation"])
        return {"text": text, "label": label}

def collate_batch(batch, tokenizer, cap_len=1024, pad_to_multiple_of=8):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)

    enc = tokenizer(
        texts,
        padding="longest",          # <-- dynamic padding to batch max length
        truncation=True,
        max_length=cap_len,         # safety cap; raise/lower as you like
        return_tensors="pt",
    )

    if pad_to_multiple_of:
        # re-pad to multiple-of-N for Tensor Cores efficiency (optional)
        pad_len = (-enc["input_ids"].shape[1]) % pad_to_multiple_of
        if pad_len:
            pad_id = tokenizer.pad_token_id
            enc["input_ids"] = torch.nn.functional.pad(enc["input_ids"], (0, pad_len), value=pad_id)
            enc["attention_mask"] = torch.nn.functional.pad(enc["attention_mask"], (0, pad_len), value=0)

    enc["labels"] = labels
    return enc

$\textbf{4: Loading the dataset}$

In [ ]:

df = pd.read_csv("./DataFolder/train.csv")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = RedditGuidelineDataset(train_df)
val_dataset = RedditGuidelineDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_batch(b, tokenizer, cap_len=768)
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_batch(b, tokenizer, cap_len=768)
)
